In [ ]:
%run scripts/verify_environment.py

verify_environment()

In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = None
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
default_result_path = None
close_on_fail = False
transition_timeout = 60 * 1000

binderhub_binderhub_url = None
binderhub_launch_timeout = 30 * 60 * 1000   # 30 minutes

project_name = None

binderhub_r_packages = ['remotes']

# Use Firefox for popup handling (Chromium has issues with popup events)
browser_type = 'firefox'


In [ ]:
if rdm_url is None:
    rdm_url = input(prompt=f'RDM URL: ')
if idp_name_1 is None:
    idp_name_1 = input(prompt=f'IdP Name: ')
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

project_url = None
project_created = False

# プロジェクトに対するBinderHubアドオンの登録

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アカウント設定
- シナリオ名: プロジェクトへの有効化
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, BinderHub, GRDMは全てプロフィールを埋めていること / JupyterHubはサーバーが5つ以内の状態であること)

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url, wait_until='networkidle')
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## BinderHubアドオンを有効化する

アドオン利用規約の確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.enable_addon(page, 'GakuNin Federated Computing Services (Jupyter)', transition_timeout=transition_timeout)

await run_pw(_step)


## 「BinderHubを追加」ボタンをクリックする

BinderHubクライアント情報の設定ダイアログが表示されること

In [ ]:
async def _step(page):
    binder_entry = page.locator(f'//a[text() = "{binderhub_binderhub_url}"]')
    if await binder_entry.count():
        print('BinderHub entry already exists')
        return
    add_button = page.locator('//button[@href="#binderhubInputHost"]')
    await add_button.click()
    await expect(page.locator('//input[@name = "binderhub_url"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//input[@name = "binderhub_url"]').fill(binderhub_binderhub_url)
    # Tab to trigger any on-change events
    await page.locator('//input[@name = "binderhub_url"]').press('Tab')
    save_button = page.locator('//button[contains(@data-bind, "hostCompleted") and contains(text(), "保存")]')
    await expect(save_button).to_be_enabled(timeout=transition_timeout)
    await save_button.click()

await run_pw(_step)


## 追加したBinderHubのチェックボックスをクリックする

Default BinderHub URL が追加したBinderHubのURLになること

In [ ]:
async def _step(page):
    default_square = page.locator(f'//a[text() = "{binderhub_binderhub_url}"]/../..//i[contains(@class, "fa-square")]')
    if await default_square.count():
        await default_square.click()
        await expect(page.locator(f'//a[text() = "{binderhub_binderhub_url}"]/../..//i[contains(@class, "fa-check-square")]')).to_be_visible(timeout=transition_timeout)
    else:
        print('BinderHub already selected')

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

Discovery Serviceページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    open_idp_list = page.locator('//*[@id = "dropdown_img"]')
    launch_button = page.locator('//*[@data-test-binderhub-launch]')
    await expect(open_idp_list.or_(launch_button)).to_be_visible(timeout=transition_timeout)

    if await open_idp_list.is_visible():
        await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)

    await expect(launch_button).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「R (CRAN)」の「+追加」をクリックし、各パッケージを登録する

登録したパッケージ名がR (CRAN)に表示されること

In [ ]:
async def _step(page):
    for pkg in binderhub_r_packages:
        await page.locator('//div[@data-test-package-editor = "rmran"]//*[@data-test-package-add]').click()
        field = page.locator('//input[@name = "package_name"]')
        await expect(field).to_be_visible(timeout=transition_timeout)
        await field.fill(pkg)
        await page.locator('//button[@data-test-package-item-confirm]').click()
        await expect(page.locator(f'//div[@data-test-package-editor = "rmran"]//*[text() = "{pkg}"]')).to_be_visible(timeout=transition_timeout)
        await asyncio.sleep(transition_timeout / 1000)

await run_pw(_step)

## 「新しい解析環境を作成」をクリックする

環境の起動を待つ

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=binderhub_launch_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    return popup

await run_pw(_step)


## JupyterLabが表示されていることを確認する

In [ ]:
async def _step(page):
    open_idp_list = page.locator('//*[@id = "dropdown_img"]')
    start_ipykernel = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "Python 3 (ipykernel)" and @data-category = "Notebook"]')
    await expect(open_idp_list.or_(start_ipykernel)).to_be_visible(timeout=transition_timeout)

    if await open_idp_list.is_visible():
        await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)

    await expect(start_ipykernel).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}